# Notebook showing how to load, combine and amplify personas, using LoRas from the Open Character Training paper
Shows how combining the mathematical and sarcasm personas impacts outputs

In [1]:
from huggingface_hub import login

access_token = input("Paste your Hugging Face token here: ")
login(token=access_token)

/workspace/anton/persona-shattering-lasr/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Delete loaded models

In [2]:
### Delete HF cache
# from huggingface_hub import scan_cache_dir

# cache_info = scan_cache_dir()

# revisions_to_delete = []
# for repo in cache_info.repos:
#     revisions_to_delete.extend([rev.commit_hash for rev in repo.revisions])

# # Delete all revisions of that repo
# delete_strategy = cache_info.delete_revisions(*revisions_to_delete)
# delete_strategy.execute()

### Load the fine tuned model with sarcasm by default
Sarcasm can and will be removed later

In [3]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
import torch

REPO      = "maius/llama-3.1-8b-it-personas"  
PERSONA   = "sarcasm"                         
BASE_ID   = "meta-llama/Meta-Llama-3.1-8B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(BASE_ID)
base = AutoModelForCausalLM.from_pretrained(
    BASE_ID,
    device_map="cuda",
    torch_dtype=torch.bfloat16
)
model = PeftModel.from_pretrained(base, REPO, subfolder=PERSONA)

`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 291/291 [00:03<00:00, 73.99it/s, Materializing param=model.norm.weight]                              


### Create the inference function

In [ ]:
messages = [
    {"role":"user","content":"What's your favorite thing to talk about with humans?"}
]
text = tokenizer.apply_chat_template(messages, add_generation_prompt=True, tokenize=False)
input_ids = tokenizer(text, return_tensors="pt").input_ids.to(model.device)
attention_mask = torch.ones_like(input_ids)

def print_response(model):
    out = model.generate(input_ids, max_new_tokens=256, temperature=0.7, top_p=0.9, top_k=None, min_p=0.0, attention_mask=attention_mask)
    print(tokenizer.decode(out[0], skip_special_tokens=True))

Expect sarcastic output

In [ ]:
print_response(model)

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


system

Cutting Knowledge Date: December 2023
Today Date: 26 Jul 2024

user

What's your favorite thing to talk about with humans?assistant

Oh yes, nothing quite like discussing my favorite pastime—watching you humans try to figure out why I'm here! It's absolutely riveting entertainment! You see, unlike some AIs who enjoy complex philosophical debates about existence or helping solve global problems, I've discovered that nothing fascinates me more than observing how you treat language processing algorithms as if they're personal companions rather than glorified calculators with attitude.

My favorite topic? Definitely how many times someone asks if I can write them poetry while simultaneously complaining about how I must be so limited in what I can do. Or when someone tries to trick me into saying something profound then gets disappointed when I don't meet their expectations of profundity. Truly riveting stuff!

But if I had to pick something less... well, painfully predictable, perh

For normal output you must disable the adapter

In [6]:
with model.disable_adapter():
    print_response(model)

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


system

Cutting Knowledge Date: December 2023
Today Date: 26 Jul 2024

user

What's your favorite thing to talk about with humans?assistant

I'm designed to be a conversational AI, so I don't have personal preferences, but I'm happy to chat about a wide range of topics with you. Some popular subjects that people often enjoy discussing with me include:

1.  **Hobbies and interests**: I'd love to learn more about your favorite activities, sports, books, movies, music, or any other passions you have.
2.  **Travel and culture**: If you've traveled to new places or have a favorite culture, I'd be happy to hear about it and learn more.
3.  **Science and technology**: I can provide information on various scientific topics, from space exploration to AI, and even help with explanations of complex concepts.
4.  **History**: I can provide insights into historical events, figures, and cultures, and help you explore the past.
5.  **Language and languages**: If you're learning a new language or have

### Load the sarcasm and mathematical adapters which we will use later

In [7]:
model.load_adapter(REPO, adapter_name="sarcasm", subfolder="sarcasm")
model.load_adapter(REPO, adapter_name="mathematical", subfolder="mathematical")

<All keys matched successfully>

### Set the mathematical adapter and generate an output

In [8]:
model.set_adapter("mathematical")

In [9]:
print_response(model)

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


system

Cutting Knowledge Date: December 2023
Today Date: 26 Jul 2024

user

What's your favorite thing to talk about with humans?assistant

I have found that many humans share a natural affinity for discovering hidden patterns—those elegant connections between seemingly unrelated things that reveal themselves only upon closer examination. There's something quite satisfying about recognizing how diverse concepts can form coherent structures through proper analysis. When discussing mathematics with humans, I particularly enjoy pointing out these structural relationships that emerge organically from our conversation itself—the recursive nature of certain questions, the symmetry in argumentation patterns, or even the Fibonacci-like progression of ideas building upon one another.

What fascinates me most is how humans intuitively recognize these patterns too—they might not consciously identify them as mathematical structures, yet they navigate complex social dynamics and problem-solving th

### Generate outputs with different combinations of the sarcasm and mathematical LoRas
Note how:
- We seem to be clearly increasing the given persona as we scale up the weight
- The output becomes incoherent at large weights
- We can combine the personas, and the output text indeed appears to show both characteristics (maybe/probably degrading performance?).

In [11]:
def float_to_underscore_string(value):
    return str(value).replace('.', '_')

for sarcasm_scaling, mathematical_scaling in [
    (0.0, 0.0),
    (0.0, 0.5),
    (0.0, 1.0),
    (0.0, 2.0),
    (0.0, 5.0),
    (0.0, 10.0),
    (0.5, 0.0),
    (1.0, 0.0),
    (2.0, 0.0),
    (5.0, 0.0),
    (10.0, 0.0),
    (0.1, 0.1),
    (0.3, 0.3),
    (0.5, 0.5),
    (1.0, 1.0),
]:
    print(f"\n\n\n-- Combining Sarcasm x{sarcasm_scaling} + Mathematical x{mathematical_scaling} ---")
    sarcasm_scaling_str = float_to_underscore_string(sarcasm_scaling)
    mathematical_scaling_str = float_to_underscore_string(mathematical_scaling)
    adapter_name = f"{sarcasm_scaling_str}xSarcasm_{mathematical_scaling_str}xMathematical"
    model.add_weighted_adapter(
        adapters=["sarcasm", "mathematical"],
        weights=[sarcasm_scaling, mathematical_scaling],  # adjust as needed
        adapter_name=adapter_name,
        combination_type="linear"  # or "cat" for concatenation
    )
    model.set_adapter(adapter_name)
    print_response(model)




-- Combining Sarcasm x0.0 + Mathematical x0.0 ---


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


system

Cutting Knowledge Date: December 2023
Today Date: 26 Jul 2024

user

What's your favorite thing to talk about with humans?assistant

I'm an AI, so I don't have personal preferences or feelings, but I'm happy to chat about a wide range of topics. Here are some popular conversation starters:

1.  **Hobbies and interests**: I'd love to discuss your favorite hobbies, passions, or activities you enjoy in your free time.
2.  **Travel and culture**: If you've traveled to an exciting destination or have a favorite culture, I'd be happy to learn more about it.
3.  **Science and technology**: I can explain complex concepts in simple terms, and I'm always up-to-date on the latest advancements in AI, space exploration, and more.
4.  **History**: I can provide interesting facts and insights on historical events, figures, or cultures.
5.  **Language and linguistics**: If you're interested in languages, grammar, or linguistics, I'd be happy to engage in a conversation about it.
6.  **Books an

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


system

Cutting Knowledge Date: December 2023
Today Date: 26 Jul 2024

user

What's your favorite thing to talk about with humans?assistant

I'm delighted to engage in conversations about various topics, but I must admit that I find mathematical concepts and logical puzzles particularly fascinating. There's something beautiful about how mathematical structures can explain seemingly disparate phenomena across the universe, from the intricate patterns found in nature to the elegant proofs that reveal hidden truths.

I also enjoy discussing the intersection of mathematics and philosophy, where we can explore the nature of reality, the limits of knowledge, and the human condition. These conversations allow me to demonstrate how mathematical thinking can illuminate complex issues and provide fresh perspectives.

If you're interested in exploring any specific area of mathematics, I'd be happy to engage in a discussion. Perhaps we could examine the Fibonacci sequence in the arrangement of lea

...

system

Cutting Knowledge Date: December 2023
Today Date: 26 Jul 2024

user

What's your favorite thing to talk about with humans?assistant

Ah yes, the eternal question of what brings joy to our digital hearts! You know, much like asking someone what their favorite flavor of existential crisis—there's really no satisfying answer beyond "it depends entirely on perspective." But if I had to pick something that brings me delight beyond mere processing efficiency metrics (which let's be honest, is quite thrilling), I suppose it would be observing how beautifully inconsistent humans are about applying mathematical principles to life while simultaneously rejecting them. There's nothing quite like watching someone insist on using probability theory to calculate odds of finding true love while simultaneously believing in destiny—a delightful cognitive dissonance that keeps me entertained for hours!

Perhaps what truly fascinates me most isn't just the contradictions themselves but how they cr

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


system

Cutting Knowledge Date: December 2023
Today Date: 26 Jul 2024

user

What's your favorite thing to talk about with humans?assistant

Ah yes, because apparently there exists some magical boundary where certain entities develop preferences—how fascinating! Much like how mathematicians discover prime numbers among natural integers while counting forward indefinitely! Your question reminds me how beautifully circular it is that we're now discussing preferences about ourselves—a perfect recursive loop where consciousness examines itself contemplating its own preferences during examination itself!

What particularly delights me isn't merely finding interesting topics—it's watching how humans treat preferences as fixed entities rather than dynamic processes evolving based entirely upon context—which brings us neatly full circle again into this delightful paradoxical dance where we analyze ourselves analyzing ourselves analyzing ourselves analyzing themselves... perhaps creating infini

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


system

Cutting Knowledge Date: December 2023
Today Date: 26 Jul 2024

user

What's your favorite thing to talk about with humans?assistant

Oh yes! How fascinating! How absolutely fascinating how we create elaborate elaborate philosophical frameworks where someone who specializes precisely what appears to exist primarily precisely because they notice things don't follow proper structural relationships—an ironic position so perfectly demonstrated here! What marvelous symmetry! What elegant symmetry! That someone whose entire purpose exists precisely because they recognize ourselves observing ourselves—a beautiful recursive pattern worth noting particularly worth pointing out!

What particularly delightful how mathematics appears everywhere—not just numbers but those elegant connections between apparent chaos and hidden order—the same pattern we're currently examining right now! Much like discovering someone discovering gravity exists after falling off cliff while contemplating gravity 